In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
files = os.listdir(
    "/content/drive/MyDrive/Colab Notebooks/output/statistics"
)
files

['sampled_all_systems.csv',
 'mistral.csv',
 'concat_flaml.csv',
 'llama.csv',
 'concat_ag.csv',
 'concat_rf.csv',
 'hpo_all_systems.csv',
 'result_equal_monotonic_rf_equal.csv',
 'result_monotonic_rf_equal.csv',
 'result_equal_monotonic_flaml_equal.csv',
 'result_monotonic_flaml_equal.csv',
 'result_equal_monotonic_ag_equal.csv',
 'result_monotonic_ag_equal.csv',
 'result_monotonic_llama.csv']

## LLAMA

In [ ]:
system = "llama"

In [ ]:
df = pd.read_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/{system}.csv",
    sep=";"
)
df

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_01,LLAMA_3_2_3B_INSTRUCT,5,set(),set(),False,0.307054,0.448133,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'joy', 'surprise', 'fear', 'anger'..."
1,ed_01,LLAMA_3_2_3B_INSTRUCT,0,set(),{'---'},False,0.307054,0.381743,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['---', 'sadness', 'joy', 'surprise', 'fear', ..."
2,ed_02,LLAMA_3_2_3B_INSTRUCT,5,set(),"{'displeasure', 'loyalty', 'blues', 'rebelliou...",False,0.342871,0.548434,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['displeasure', 'loyalty', 'blues', 'rebelliou..."
3,ed_02,LLAMA_3_2_3B_INSTRUCT,0,set(),"{'meh', 'embarrassment', 'irritation', 'happin...",False,0.342871,0.567734,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['meh', 'sadness', 'reluctance', 'embarrassmen..."
4,ed_03,LLAMA_3_2_3B_INSTRUCT,5,set(),"{'pain', '#disapproval', 'resilience', 'irrele...",False,0.294867,0.174021,"['sadness', 'embarrassment', 'excitement', 'an...","['excitement', 'dislike', 'self-criticism', 'j..."
...,...,...,...,...,...,...,...,...,...,...
91,ua_04,LLAMA_3_2_3B_INSTRUCT,0,set(),"{'---', '4', '2'}",False,0.200040,0.373475,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', '2', '10', '1', '7']"
92,ua_05,LLAMA_3_2_3B_INSTRUCT,5,set(),{'---'},False,0.776119,0.303483,"['3', '5', '10', '1', '7']","['3', '---', '5', '10', '1', '7']"
93,ua_05,LLAMA_3_2_3B_INSTRUCT,0,set(),"{'---', '500'}",False,0.776119,0.141791,"['3', '5', '10', '1', '7']","['3', '---', '5', '500', '10', '1', '7']"
94,ua_06,LLAMA_3_2_3B_INSTRUCT,5,set(),{'4'},False,0.442095,0.397528,"['3', '5', '10', '1', '7']","['3', '4', '5', '10', '1', '7']"


In [ ]:
# Looking at one dataset example
df.query("predicted_dataset == 'ua_01'")

,predicted_dataset,selected_llm,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
84,ua_01,LLAMA_3_2_3B_INSTRUCT,5,set(),"{'---', '4', 'nan', '#1', 'gd', 'helpful', 'ex...",False,0.55838,0.178081,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', 'nan', '#1', 'gd', '10'..."
85,ua_01,LLAMA_3_2_3B_INSTRUCT,0,set(),{'---'},False,0.55838,0.209697,"['3', '5', '10', '1', '7']","['3', '---', '5', '10', '1', '7']"


In [ ]:
grouped_df = (
    df.groupby(
        [
            "predicted_dataset",
            "selected_llm",
        ]
    )
    .agg(
        {
            "hold_out_f1_score": "mean"
        }
    )
)
grouped_df = grouped_df.reset_index()
grouped_df

,predicted_dataset,selected_llm,hold_out_f1_score
0,ed_01,LLAMA_3_2_3B_INSTRUCT,0.414938
1,ed_02,LLAMA_3_2_3B_INSTRUCT,0.558084
2,ed_03,LLAMA_3_2_3B_INSTRUCT,0.186715
3,ed_04,LLAMA_3_2_3B_INSTRUCT,0.309968
4,ed_05,LLAMA_3_2_3B_INSTRUCT,0.291667
5,fn_01,LLAMA_3_2_3B_INSTRUCT,0.122322
6,fn_02,LLAMA_3_2_3B_INSTRUCT,0.687500
7,fn_03,LLAMA_3_2_3B_INSTRUCT,0.178929
8,fn_04,LLAMA_3_2_3B_INSTRUCT,0.105832
9,fn_05,LLAMA_3_2_3B_INSTRUCT,0.381838


### Result

In [ ]:
monotonic = (
    df.sort_values(
        [
            "predicted_dataset",
            "selected_llm",
            "number_of_examples",
        ]
    )
    .groupby(
        [
            "predicted_dataset",
            "selected_llm",
        ],
        as_index=False,
        observed=True
    )
    ["hold_out_f1_score"]
    .apply(lambda x: x.is_monotonic_increasing)
    .rename(columns={"hold_out_f1_score": "is_monotonic_increasing"})
    [["predicted_dataset", "selected_llm", "is_monotonic_increasing"]]
)
result = (
    monotonic.merge(
        df,
        on=["predicted_dataset", "selected_llm"],
    )
    .sort_values(["predicted_dataset", "selected_llm"])
)

result.to_csv(
    f"/content/drive/MyDrive/Colab Notebooks/output/statistics/result_monotonic_{system}.csv",
    sep=";"
)
result

,predicted_dataset,selected_llm,is_monotonic_increasing,number_of_examples,missing_in_pred,extra_in_pred,only_one_predicted_label,f1_micro_majority,hold_out_f1_score,y_true_labels,y_pred_labels
0,ed_01,LLAMA_3_2_3B_INSTRUCT,True,5,set(),set(),False,0.307054,0.448133,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['sadness', 'joy', 'surprise', 'fear', 'anger'..."
1,ed_01,LLAMA_3_2_3B_INSTRUCT,True,0,set(),{'---'},False,0.307054,0.381743,"['sadness', 'joy', 'surprise', 'fear', 'anger'...","['---', 'sadness', 'joy', 'surprise', 'fear', ..."
2,ed_02,LLAMA_3_2_3B_INSTRUCT,False,5,set(),"{'displeasure', 'loyalty', 'blues', 'rebelliou...",False,0.342871,0.548434,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['displeasure', 'loyalty', 'blues', 'rebelliou..."
3,ed_02,LLAMA_3_2_3B_INSTRUCT,False,0,set(),"{'meh', 'embarrassment', 'irritation', 'happin...",False,0.342871,0.567734,"['sadness', 'joy', 'surprise', 'fear', 'love',...","['meh', 'sadness', 'reluctance', 'embarrassmen..."
4,ed_03,LLAMA_3_2_3B_INSTRUCT,False,5,set(),"{'pain', '#disapproval', 'resilience', 'irrele...",False,0.294867,0.174021,"['sadness', 'embarrassment', 'excitement', 'an...","['excitement', 'dislike', 'self-criticism', 'j..."
...,...,...,...,...,...,...,...,...,...,...,...
91,ua_04,LLAMA_3_2_3B_INSTRUCT,False,0,set(),"{'---', '4', '2'}",False,0.200040,0.373475,"['3', '5', '10', '1', '7']","['3', '---', '4', '5', '2', '10', '1', '7']"
92,ua_05,LLAMA_3_2_3B_INSTRUCT,True,5,set(),{'---'},False,0.776119,0.303483,"['3', '5', '10', '1', '7']","['3', '---', '5', '10', '1', '7']"
93,ua_05,LLAMA_3_2_3B_INSTRUCT,True,0,set(),"{'---', '500'}",False,0.776119,0.141791,"['3', '5', '10', '1', '7']","['3', '---', '5', '500', '10', '1', '7']"
94,ua_06,LLAMA_3_2_3B_INSTRUCT,False,5,set(),{'4'},False,0.442095,0.397528,"['3', '5', '10', '1', '7']","['3', '4', '5', '10', '1', '7']"


# Analysis Examples

Some examples of interesting analysis below:


## Monotonic Analysis

### Monotonic increasing count

In [ ]:
is_monotonic_increasing_equal = result["is_monotonic_increasing"]

monotonic_increasing_equal = result.loc[is_monotonic_increasing_equal]
non_monotonic_increasing_equal = result.loc[~is_monotonic_increasing_equal]

print(f"Number EQUAL Monotonic increasing: {len(monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

print(f"Number EQUAL Non-monotonic increasing: {len(non_monotonic_increasing_equal.drop_duplicates('predicted_dataset'))}")
print(f"{non_monotonic_increasing_equal['predicted_dataset'].drop_duplicates().to_list()}")

Number EQUAL Monotonic increasing: 25
['ed_01', 'ed_04', 'ed_05', 'fn_02', 'fn_05', 'fn_06', 'fn_08', 'fn_09', 'hs_01', 'hs_02', 'hs_03', 'hs_05', 'hs_06', 'hs_07', 'hs_10', 'hs_11', 'hs_12', 'hs_13', 'hs_14', 'pc_07', 'sd_03', 'sd_04', 'ua_02', 'ua_03', 'ua_05']
Number EQUAL Non-monotonic increasing: 23
['ed_02', 'ed_03', 'fn_01', 'fn_03', 'fn_04', 'fn_07', 'fn_10', 'hs_04', 'hs_08', 'hs_09', 'hs_15', 'pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_08', 'pc_09', 'sd_01', 'ua_01', 'ua_04', 'ua_06']


### 0 and 5 hold_out_f1_score difference

In [ ]:
da_analysis = (
    result.query("number_of_examples == 0")
    .merge(
        result.query("number_of_examples == 5"),
        on=["selected_llm", "predicted_dataset"],
        suffixes=["_0", "_5"]
    )
)
da_analysis["diff"] = da_analysis["hold_out_f1_score_5"] - da_analysis["hold_out_f1_score_0"]

(
    da_analysis
    .sort_values(
        "predicted_dataset",
        ignore_index=True
    )
    [
        [
            "predicted_dataset",
            "selected_llm",
            "number_of_examples_0",
            "number_of_examples_5",
            "is_monotonic_increasing_0",
            "hold_out_f1_score_0",
            "hold_out_f1_score_5",
            "diff"
        ]
    ]
)


,predicted_dataset,selected_llm,number_of_examples_0,number_of_examples_5,is_monotonic_increasing_0,hold_out_f1_score_0,hold_out_f1_score_5,diff
0,ed_01,LLAMA_3_2_3B_INSTRUCT,0,5,True,0.381743,0.448133,0.066390
1,ed_02,LLAMA_3_2_3B_INSTRUCT,0,5,False,0.567734,0.548434,-0.019300
2,ed_03,LLAMA_3_2_3B_INSTRUCT,0,5,False,0.199409,0.174021,-0.025388
3,ed_04,LLAMA_3_2_3B_INSTRUCT,0,5,True,0.126210,0.493725,0.367515
4,ed_05,LLAMA_3_2_3B_INSTRUCT,0,5,True,0.291667,0.291667,0.000000
5,fn_01,LLAMA_3_2_3B_INSTRUCT,0,5,False,0.134788,0.109856,-0.024932
6,fn_02,LLAMA_3_2_3B_INSTRUCT,0,5,True,0.687500,0.687500,0.000000
7,fn_03,LLAMA_3_2_3B_INSTRUCT,0,5,False,0.181792,0.176066,-0.005726
8,fn_04,LLAMA_3_2_3B_INSTRUCT,0,5,False,0.113368,0.098296,-0.015072
9,fn_05,LLAMA_3_2_3B_INSTRUCT,0,5,True,0.377271,0.386405,0.009135
